In [ ]:
import numpy as np
import pandas as pd
import io
# Données SIRD simplifiées
data = ("Date,I,S,R,D\n1/3/2020,0,8250000,0,0\n8/3/2020,21,8249976,3,0\n15/3/2020,1034,8248142,820,4\n"
       "22/3/2020,2580,8245613,1755,52\n29/3/2020,3560,8244389,1695,356\n5/4/2020,3782,8243842,1618,758\n"
       "12/4/2020,2887,8245177,1182,754\n19/4/2020,2342,8245852,1312,494\n26/4/2020,1003,8248301,396,300\n"
       "3/5/2020,787,8248642,361,210\n10/5/2020,457,8249191,238,114\n17/5/2020,360,8249360,191,89\n"
       "24/5/2020,467,8249152,327,54\n31/5/2020,230,8249600,128,42\n7/6/2020,197,8249660,114,29\n"
       "14/6/2020,180,8249687,111,22\n21/6/2020,172,8249700,106,22\n28/6/2020,201,8249630,157,12\n"
       "5/7/2020,155,8249719,117,9\n12/7/2020,223,8249580,190,7\n19/7/2020,175,8249688,127,10\n"
       "26/7/2020,140,8249745,107,8\n2/8/2020,135,8249763,100,2\n9/8/2020,133,8249760,104,3\n"
       "16/8/2020,140,8249741,114,5\n23/8/2020,150,8249723,124,3\n30/8/2020,186,8249648,160,6\n"
       "6/9/2020,207,8249617,170,6\n13/9/2020,263,8249491,243,3\n20/9/2020,206,8249298,491,5\n"
       "27/9/2020,382,8249273,341,4\n4/10/2020,335,8249374,284,7\n11/10/2020,281,8249487,225,7\n"
       "18/10/2020,412,8249216,368,4\n25/10/2020,362,8249320,308,10\n1/11/2020,501,8249053,434,12\n"
       "8/11/2020,788,8248459,744,9\n15/11/2020,896,8248300,790,14\n22/11/2020,1148,8247820,1020,12\n"
       "29/11/2020,1891,8246981,1113,15\n6/12/2020,1753,8246667,1558,22\n13/12/2020,2177,8245873,1918,32\n"
       "20/12/2020,2346,8245554,2066,34\n27/12/2020,2888,8244465,2599,48\n3/1/2021,3464,8243312,3181,43\n"
       "10/1/2021,3197,8237916,8831,56")

# Chargement des données dans un DataFrame
df = pd.read_csv(io.StringIO(data), names=["Date", "I", "S", "R", "D"], header=0)

# Listes pour stocker les taux hebdomadaires
a_values = []  # Taux de guérison
b_values = []  # Taux de mortalité
r_values = []  # Taux de transmission

# Calculs
for i in range(1, len(df)):
    delta_R = df.loc[i, 'R'] - df.loc[i-1, 'R']
    delta_D = df.loc[i, 'D'] - df.loc[i-1, 'D']
    avg_I = (df.loc[i, 'I'] + df.loc[i-1, 'I']) / 2
    if avg_I > 0:
        a_weekly = delta_R / avg_I
        b_weekly = delta_D / avg_I
        if 0 <= a_weekly <= 1:
            a_values.append(a_weekly)
        if 0 <= b_weekly <= 1:
            b_values.append(b_weekly)
        S_avg = (df.loc[i, 'S'] + df.loc[i-1, 'S']) / 2
        delta_I = df.loc[i, 'I'] - df.loc[i-1, 'I']
        if S_avg > 0:
            r_weekly = (delta_I + (a_weekly + b_weekly) * avg_I) / (S_avg * avg_I)
            if 0 <= r_weekly <= 1e-5:
                r_values.append(r_weekly)

# Moyennes des paramètres
a_mean = np.mean(a_values)
b_mean = np.mean(b_values)
r_mean = np.mean(r_values)

# Affichage des résultats
print("Paramètres du modèle SIRD :")
print(f"a (taux de guérison) = {a_mean:.6f}")
print(f"b (taux de mortalité) = {b_mean:.6f}")
print(f"r (taux de transmission) = {r_mean:.8e}")

Paramètres du modèle SIRD :
a (taux de guérison) = 0.217206
b (taux de mortalité) = 0.015728
r (taux de transmission) = 7.92747107e-08


In [ ]:
import numpy as np

# Définition du polynôme I(t)
def I(t):
    return (-2.32412098 - 69.61859738*t + 10.2597809*t**2 - 0.17044681*t**3
            + 0.05194685*t**4 - 0.00470237*t**5 + 0.00018242*t**6
            - 4.01177364e-6*t**7 + 5.55325441e-8*t**8 - 4.99678311e-10*t**9
            + 2.84871774e-12*t**10 - 8.94417282e-15*t**11 + 5.21179435e-18*t**12
            + 5.46889091e-20*t**13 - 7.78089532e-23*t**14 - 3.86851168e-25*t**15
            + 3.24417332e-28*t**16 + 3.10731994e-30*t**17 + 2.35051145e-33*t**18
            - 1.7908815e-35*t**19 - 5.54546914e-38*t**20 - 4.6616709e-42*t**21
            + 3.88830445e-43*t**22 + 1.1351279e-45*t**23 + 5.7623148e-49*t**24
            - 6.4762595e-51*t**25 - 2.54994305e-53*t**26 - 3.90839925e-56*t**27
            + 5.10386316e-59*t**28 + 4.72621443e-61*t**29 + 1.35924196e-63*t**30
            + 1.57220143e-66*t**31 - 4.16127271e-69*t**32 - 2.78581668e-71*t**33
            - 7.76920025e-74*t**34 - 9.63341341e-77*t**35 + 2.0525657e-79*t**36
            + 1.5560816e-81*t**37 + 4.69614608e-84*t**38 + 6.99373468e-87*t**39
            - 8.08534603e-90*t**40 - 8.70360813e-92*t**41 - 2.86465056e-94*t**42
            - 4.5361581e-97*t**43 + 5.00763336e-100*t**44 + 5.62293576e-102*t**45
            + 1.71644688e-104*t**46 + 1.54412042e-107*t**47 - 9.52882385e-110*t**48
            - 4.38556362e-112*t**49 - 2.24958798e-115*t**50 + 4.40969761e-117*t**51
            - 4.76273635e-120*t**52)

# Première dérivée I'(t)
def dI_dt(t):
    return (-69.61859738 + 2*10.2597809*t - 3*0.17044681*t**2 + 4*0.05194685*t**3
            - 5*0.00470237*t**4 + 6*0.00018242*t**5 - 7*4.01177364e-6*t**6
            + 8*5.55325441e-8*t**7 - 9*4.99678311e-10*t**8 + 10*2.84871774e-12*t**9
            - 11*8.94417282e-15*t**10 + 12*5.21179435e-18*t**11 + 13*5.46889091e-20*t**12
            - 14*7.78089532e-23*t**13 - 15*3.86851168e-25*t**14 + 16*3.24417332e-28*t**15
            + 17*3.10731994e-30*t**16 + 18*2.35051145e-33*t**17 - 19*1.7908815e-35*t**18
            - 20*5.54546914e-38*t**19 - 21*4.6616709e-42*t**20 + 22*3.88830445e-43*t**21
            + 23*1.1351279e-45*t**22 + 24*5.7623148e-49*t**23 - 25*6.4762595e-51*t**24
            - 26*2.54994305e-53*t**25 - 27*3.90839925e-56*t**26 + 28*5.10386316e-59*t**27
            + 29*4.72621443e-61*t**28 + 30*1.35924196e-63*t**29 + 31*1.57220143e-66*t**30
            - 32*4.16127271e-69*t**31 - 33*2.78581668e-71*t**32 - 34*7.76920025e-74*t**33
            - 35*9.63341341e-77*t**34 + 36*2.0525657e-79*t**35 + 37*1.5560816e-81*t**36
            + 38*4.69614608e-84*t**37 + 39*6.99373468e-87*t**38 - 40*8.08534603e-90*t**39
            - 41*8.70360813e-92*t**40 - 42*2.86465056e-94*t**41 - 43*4.5361581e-97*t**42
            + 44*5.00763336e-100*t**43 + 45*5.62293576e-102*t**44 + 46*1.71644688e-104*t**45
            + 47*1.54412042e-107*t**46 - 48*9.52882385e-110*t**47 - 49*4.38556362e-112*t**48
            - 50*2.24958798e-115*t**49 + 51*4.40969761e-117*t**50 - 52*4.76273635e-120*t**51)

# Méthode de Newton simplifiée
def Newton(x0, f, df, epsilon):
    x = x0
    y = x - f(x) / df(x)
    while abs(y - x) > epsilon:
        x = y
        y = x - f(x) / df(x)
    return y

# Recherche d’un bon point de départ dans [0, 30]
t_values = np.linspace(0, 30, 1000)
I_values = [I(t) for t in t_values]
t0 = t_values[np.argmax(I_values)]  # Point où I est maximal approximativement

# Utilisation de Newton pour trouver le pic précisément
t_pic = Newton(t0, dI_dt, lambda t: (dI_dt(t + 1e-5) - dI_dt(t)) / 1e-5, 1e-6)  # Dérivée approximée

# Affichage du résultat
print(f"Le pic épidémique est atteint à t = {t_pic:.6f}")
print(f"Valeur de I(t) au pic : {I(t_pic):.6f}")
print(f"dI/dt au pic : {dI_dt(t_pic):.2e}")

Le pic épidémique est atteint à t = 32.284660
Valeur de I(t) au pic : 3775.694882
dI/dt au pic : 5.20e-12


In [ ]:
import numpy as np
from datetime import datetime, timedelta

# Définition du polynôme I(t)
def f(t):
    return -2.32412098 - 69.61859738*t + 10.2597809*t**2 - 0.17044681*t**3 + 0.05194685*t**4 - 0.00470237*t**5 + 0.00018242*t**6 - 4.01177364e-6*t**7 + 5.55325441e-8*t**8 - 4.99678311e-10*t**9 + 2.84871774e-12*t**10 - 8.94417282e-15*t**11 + 5.21179435e-18*t**12 + 5.46889091e-20*t**13 - 7.78089532e-23*t**14 - 3.86851168e-25*t**15 + 3.24417332e-28*t**16 + 3.10731994e-30*t**17 + 2.35051145e-33*t**18 - 1.7908815e-35*t**19 - 5.54546914e-38*t**20 - 4.6616709e-42*t**21 + 3.88830445e-43*t**22 + 1.1351279e-45*t**23 + 5.7623148e-49*t**24 - 6.4762595e-51*t**25 - 2.54994305e-53*t**26 - 3.90839925e-56*t**27 + 5.10386316e-59*t**28 + 4.72621443e-61*t**29 + 1.35924196e-63*t**30 + 1.57220143e-66*t**31 - 4.16127271e-69*t**32 - 2.78581668e-71*t**33 - 7.76920025e-74*t**34 - 9.63341341e-77*t**35 + 2.0525657e-79*t**36 + 1.5560816e-81*t**37 + 4.69614608e-84*t**38 + 6.99373468e-87*t**39 - 8.08534603e-90*t**40 - 8.70360813e-92*t**41 - 2.86465056e-94*t**42 - 4.5361581e-97*t**43 + 5.00763336e-100*t**44 + 5.62293576e-102*t**45 + 1.71644688e-104*t**46 + 1.54412042e-107*t**47 - 9.52882385e-110*t**48 - 4.38556362e-112*t**49 - 2.24958798e-115*t**50 + 4.40969761e-117*t**51 - 4.76273635e-120*t**52

# Paramètres
Imax = 3776  # Valeur critique
start_date = datetime(2020, 3, 1)  # Date de référence (t=0)

# Méthode de dichotomie pour trouver tc
def dichotomie(a, b, eps=0.01):
    while (b - a) > eps:
        mid = (a + b) / 2
        if f(mid) > Imax:
            b = mid
        else:
            a = mid
    return (a + b) / 2

# Recherche du temps critique
tcritique = dichotomie(20,50)
date_critique = start_date + timedelta(days=int(tcritique))

# Résultats finaux
print(f"Temps critique: {tcritique:.2f} jours")
print(f"Date critique: {date_critique.strftime('%d/%m/%Y')}")
print(f"Nombre d'infectés au temps critique: {f(tcritique):.2f}")

Temps critique: 50.00 jours
Date critique: 19/04/2020
Nombre d'infectés au temps critique: 1862.02
